# Nagi Studio — mediapipe
ジョブZIPをアップロードして実行し、結果ZIPを制作画面に取り込む。MoGe3はGPU推奨。既定の公式重みは初回約1.48GBダウンロード。手持ちの重みパスも指定可能。人物推定はアニメで未検出になる場合があり、検出を捏造しない。

In [ ]:
USE_DRIVE=True
CHECKPOINT='Ruicheng/moge-3-vitl'
DEVICE='cuda'
RESOLUTION_LEVEL=5
REFINE_STEPS=3


In [ ]:
import sys,subprocess,json,zipfile,io
from pathlib import Path
from google.colab import files,drive
if USE_DRIVE:
    drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/NagiStudio-Compute' if USE_DRIVE else '/content/nagi-compute')
ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','mediapipe==0.10.32'],check=True)

In [ ]:
workers={'run_depth_job.py': '"""MoGe-3 local/Colab worker with explicit checkpoint and validity masks."""\nimport argparse\nimport io\nimport json\nimport time\nfrom pathlib import Path\nimport zipfile\nimport hashlib\nimport numpy as np\nfrom PIL import Image\n\nMOGE_COMMIT = \'74fbce054ebed49800de42d0ad0e83495065719a\'\nMODEL_ID = \'Ruicheng/moge-3-vitl\'\nMODEL_REVISION = \'184008f877d7ad1ad4c2cd2182a9bd1f63d0e5be\'\n\n\ndef read_job(path, backend=\'moge3\'):\n    with zipfile.ZipFile(path) as z:\n        if len(z.infolist())!=2 or set(z.namelist())!={\'job.json\',\'source.png\'} or sum(i.file_size for i in z.infolist())>100_000_000:\n            raise ValueError(\'Invalid job archive\')\n        job=json.loads(z.read(\'job.json\'));raw=z.read(\'source.png\')\n    if job.get(\'schemaVersion\')!=2 or job.get(\'backend\')!=backend or hashlib.sha256(raw).hexdigest()!=job.get(\'inputSha256\'):\n        raise ValueError(\'Job schema/backend/input hash mismatch\')\n    im=Image.open(io.BytesIO(raw))\n    if im.width*im.height>20_000_000:raise ValueError(\'Image too large\')\n    return job,im.convert(\'RGBA\')\n\n\ndef package(job, output_path, metadata, arrays):\n    meta={\'schemaVersion\':2,\'status\':\'success\',\'backend\':job[\'backend\'],\'jobId\':job[\'id\'],\n          \'projectId\':job[\'projectId\'],\'inputSha256\':job[\'inputSha256\'],**metadata,\'files\':{}}\n    out=Path(output_path);out.parent.mkdir(parents=True,exist_ok=True);temp=out.with_suffix(\'.tmp\')\n    with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for name,array in arrays.items():\n            b=io.BytesIO();np.save(b,array,allow_pickle=False);raw=b.getvalue()\n            z.writestr(name,raw);meta[\'files\'][name]=hashlib.sha256(raw).hexdigest()\n        z.writestr(\'result.json\',json.dumps(meta,ensure_ascii=False))\n    temp.replace(out)\n\n\ndef run(input_path, output_path, checkpoint=None, device=\'cuda\', resolution_level=5, refine_steps=3):\n    job,image=read_job(input_path)\n    official=not checkpoint or checkpoint == MODEL_ID\n    if official:\n        print(\'Fetching pinned official MoGe-3 checkpoint...\',flush=True)\n        from huggingface_hub import hf_hub_download\n        checkpoint=hf_hub_download(MODEL_ID,\'model.pt\',revision=MODEL_REVISION)\n    if not Path(checkpoint).is_file():raise ValueError(\'Checkpoint file not found\')\n    import torch\n    from moge.model.v3 import MoGeModel\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'CUDA unavailable. Select --device cpu or Colab GPU.\')\n    start=time.monotonic()\n    print(\'Loading MoGe-3 on \'+device,flush=True)\n    background=Image.new(\'RGBA\',image.size,\'white\');background.alpha_composite(image)\n    model=MoGeModel.from_pretrained(checkpoint).to(device).eval()\n    tensor=torch.tensor(np.asarray(background.convert(\'RGB\')).copy(),dtype=torch.float32,device=device).permute(2,0,1)/255\n    with torch.inference_mode():\n        print(f\'Inferring {image.width}x{image.height}, level={resolution_level}, refinement={refine_steps}\',flush=True)\n        result=model.infer(tensor,resolution_level=resolution_level,refine_steps=refine_steps,use_fp16=False)\n    arrays={k:result[k].detach().cpu().numpy() for k in (\'depth\',\'mask\',\'points\',\'intrinsics\')}\n    valid=arrays[\'mask\'].astype(bool)&np.isfinite(arrays[\'depth\'])&(arrays[\'depth\']>0)&(np.asarray(image)[:,:,3]>0)\n    if valid.shape!=(image.height,image.width) or not valid.any():raise ValueError(\'No valid foreground depth\')\n    arrays[\'mask\']=valid\n    arrays[\'depth\']=np.where(valid,arrays[\'depth\'],0).astype(\'float32\')\n    arrays[\'points\']=np.where(valid[:,:,None],arrays[\'points\'],0).astype(\'float32\')\n    for key,value in arrays.items():\n        if not np.isfinite(value).all():raise ValueError(\'Non-finite \'+key)\n    package(job,output_path,{\'inferred\':True,\'sourceSize\':list(image.size),\'checkpointName\':Path(checkpoint).name,\n        \'torch\':torch.__version__,\'device\':device,\'gpu\':torch.cuda.get_device_name() if device==\'cuda\' else None,\n        \'elapsedSeconds\':time.monotonic()-start,\'resolutionLevel\':resolution_level,\'refineSteps\':refine_steps,\'modelRevision\':MODEL_REVISION if official else None},\n        {k+\'.npy\':v for k,v in arrays.items()})\n    print(\'Result saved: \'+str(output_path),flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'job\');p.add_argument(\'result\');p.add_argument(\'--checkpoint\',default=MODEL_ID)\n    p.add_argument(\'--device\',choices=[\'cuda\',\'cpu\'],default=\'cuda\');p.add_argument(\'--resolution-level\',type=int,choices=range(10),default=5)\n    p.add_argument(\'--refine-steps\',type=int,choices=range(11),default=3)\n    a=p.parse_args();run(a.job,a.result,a.checkpoint,a.device,a.resolution_level,a.refine_steps)\n', 'run_landmark_job.py': '"""Image-mode MediaPipe person/face/hand inference; missing detections stay missing."""\nimport argparse\nfrom pathlib import Path\nimport urllib.request\nimport numpy as np\nfrom run_depth_job import read_job,package\n\nMODELS={\n \'pose\':\'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task\',\n \'face\':\'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task\',\n \'hand\':\'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task\'}\n\n\ndef run(job_path,result_path,models):\n    import mediapipe as mp\n    from mediapipe.tasks.python import BaseOptions\n    from mediapipe.tasks.python import vision\n    from PIL import Image\n    job,image=read_job(job_path,\'mediapipe\');root=Path(models);root.mkdir(parents=True,exist_ok=True)\n    for key,url in MODELS.items():\n        path=root/(key+\'.task\')\n        if not path.is_file():\n            temp=path.with_suffix(\'.tmp\');urllib.request.urlretrieve(url,temp);temp.replace(path)\n    bg=Image.new(\'RGBA\',image.size,\'white\');bg.alpha_composite(image)\n    rgb=mp.Image(image_format=mp.ImageFormat.SRGB,data=np.asarray(bg.convert(\'RGB\')).copy())\n    arrays={};metadata={\'inferred\':True,\'sourceSize\':list(image.size),\'warnings\':[],\'backendVersion\':mp.__version__}\n    def landmarks(values):\n        return [[p.x,p.y,p.z,float(p.visibility or 0),float(p.presence or 0)] for p in values]\n    with vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=str(root/\'pose.task\')),output_segmentation_masks=True,num_poses=1,min_pose_detection_confidence=.2,min_pose_presence_confidence=.2)) as detector:\n        result=detector.detect(rgb)\n        metadata[\'pose\']=landmarks(result.pose_landmarks[0]) if result.pose_landmarks else []\n        if metadata[\'pose\']:\n            pose=np.asarray(metadata[\'pose\'])\n            if np.linalg.norm(pose[11,:2]-pose[12,:2])<.04 or np.ptp(pose[:25,1])<.12:\n                metadata[\'rejectedPose\']=metadata[\'pose\'];metadata[\'pose\']=[]\n                metadata[\'warnings\'].append(\'Collapsed pose geometry rejected. Set shoulder/elbow/wrist joints manually.\')\n        if result.segmentation_masks:\n            mask=result.segmentation_masks[0].numpy_view().copy()\n            if mask.ndim==3 and mask.shape[2]==1:mask=mask[:,:,0]\n            arrays[\'person-mask.npy\']=mask\n            if mask.max()<.5:metadata[\'warnings\'].append(\'Person mask confidence is below 0.5; do not use it to erase the source without review.\')\n    with vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=str(root/\'face.task\')),num_faces=1,output_face_blendshapes=True)) as detector:\n        result=detector.detect(rgb);metadata[\'face\']=landmarks(result.face_landmarks[0]) if result.face_landmarks else []\n    with vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(base_options=BaseOptions(model_asset_path=str(root/\'hand.task\')),num_hands=2,min_hand_detection_confidence=.3,min_hand_presence_confidence=.3)) as detector:\n        result=detector.detect(rgb);metadata[\'hands\']=[landmarks(x) for x in result.hand_landmarks]\n    for key in (\'pose\',\'face\',\'hands\'):\n        if not metadata[key]:metadata[\'warnings\'].append(key+\' was not detected; correct manually. Anime accuracy is not guaranteed.\')\n    package(job,result_path,metadata,arrays)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'job\');p.add_argument(\'result\');p.add_argument(\'--models\',default=\'models/mediapipe\')\n    a=p.parse_args();run(a.job,a.result,a.models)\n'}
for name,source in workers.items():
    Path('/content',name).write_text(source,encoding='utf-8')
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('ジョブZIPを1つ選択してください')
JOB=ROOT/'job.zip'
JOB.write_bytes(next(iter(uploaded.values())))
sys.path.insert(0,'/content')
from run_depth_job import read_job
job,_=read_job(JOB,'mediapipe')
RESULT=ROOT/(job['id']+'-result.zip')


In [ ]:
command=[sys.executable,'/content/run_landmark_job.py',str(JOB),str(RESULT)]
command+=['--models',str(ROOT/'models')]
with (ROOT/(job['id']+'.log')).open('w') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
    if process.wait()!=0:raise RuntimeError('推論失敗。保存したログを確認してください')
files.download(str(RESULT))
